In [3]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

TRAIN = ROOT / "data/raw/train"
VALIDATION = ROOT / "data/raw/validation"
TEST = ROOT / "data/raw/test"

transactions = pd.read_parquet(TRAIN / "transactions.parquet")
transfers = pd.read_parquet(TRAIN / "transfers.parquet")
blocks = pd.read_parquet(TRAIN / "blocks.parquet")
rates = pd.read_csv(TRAIN / "market_rates.csv")
labels = pd.read_csv(TRAIN / "labels.csv")

for name, df in {
    "transactions": transactions,
    "transfers": transfers,
    "blocks": blocks,
    "rates": rates,
    "labels": labels
}.items():
    print(f"\n{name}: {df.shape}")
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isna().sum().sum())


transactions: (40131, 26)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_HASH', 'TRANSACTION_INDEX', 'ACCOUNT', 'TRANSACTION_TYPE', 'SOURCE_FEE', 'FEE', 'SEQUENCE', 'ACCOUNT_TRANSACTION_ID', 'FLAGS', 'LAST_LEDGER_SEQUENCE', 'MEMOS', 'SIGNERS', 'SOURCE_TAG', 'SIGNING_PUB_KEY', 'TICKET_SEQUENCE', 'TXN_SIGNATURE', 'TRANSACTION_RESULT', 'DELIVERED_AMOUNT', 'SUCCESS', 'EXTRA_TRANSACTION_FIELDS', 'DATE_CREATED', 'DATE_UPDATED']
Missing values: 280917

transfers: (60557, 16)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_INDEX', 'TRANSACTION_HASH', 'TRANSFER_TYPE', 'TRANSFER_INDEX', 'FROM_ADDRESS', 'TO_ADDRESS', 'TOKEN_ADDRESS', 'SYMBOL', 'SOURCE_VALUE', 'VALUE', 'DATE_CREATED', 'DATE_UPDATED']
Missing values: 0

blocks: (151200, 17)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_HASH', 'VALIDATED', 'ACCOUNT_HASH', 'TOTAL_COINS', 'CLOSE_FLAGS', 'CLOSE_TIME', 'CLOSE_TIME_HUMAN', 'CLOSE_TIME_RESOLUT

In [4]:
# 1. Missing values per column
print("=== TRANSACTIONS: MISSING VALUES ===")
print(
    transactions.isna().sum()
    .sort_values(ascending=False)
    .to_string()
)

print("\n=== LABELS: MISSING VALUES ===")
print(labels.isna().sum().to_string())

# 2. Duplicate checks
print("\n=== DUPLICATES ===")
print("Transactions duplicate hashes:",
      transactions["TRANSACTION_HASH"].duplicated().sum())

print("Labels duplicate hashes:",
      labels["TRANSACTION_HASH"].duplicated().sum())

print("Transfers duplicate rows:",
      transfers.duplicated().sum())

# 3. Target distribution
print("\n=== TARGET DISTRIBUTION ===")
print(labels["is_suspicious"].value_counts(dropna=False))
print(labels["is_suspicious"].value_counts(normalize=True, dropna=False))

# 4. Transaction types
print("\n=== TRANSACTION TYPES ===")
print(transactions["TRANSACTION_TYPE"].value_counts(dropna=False))

# 5. Transfer counts per transaction
print("\n=== TRANSFERS PER TRANSACTION ===")
print(transfers.groupby("TRANSACTION_HASH").size().describe())

=== TRANSACTIONS: MISSING VALUES ===
MEMOS                       40131
TXN_SIGNATURE               40131
TICKET_SEQUENCE             40131
SIGNING_PUB_KEY             40131
SOURCE_TAG                  40131
ACCOUNT_TRANSACTION_ID      40131
SIGNERS                     40131
LEDGER_INDEX                    0
DATE_CREATED                    0
EXTRA_TRANSACTION_FIELDS        0
SUCCESS                         0
DELIVERED_AMOUNT                0
TRANSACTION_RESULT              0
LAST_LEDGER_SEQUENCE            0
TIMESTAMP                       0
FLAGS                           0
SEQUENCE                        0
FEE                             0
SOURCE_FEE                      0
TRANSACTION_TYPE                0
ACCOUNT                         0
TRANSACTION_INDEX               0
TRANSACTION_HASH                0
LEDGER_HASH                     0
DATETIME                        0
DATE_UPDATED                    0

=== LABELS: MISSING VALUES ===
TRANSACTION_HASH        0
is_suspicious        

In [5]:
columns_to_drop = [
    "MEMOS",
    "TXN_SIGNATURE",
    "TICKET_SEQUENCE",
    "SIGNING_PUB_KEY",
    "SOURCE_TAG",
    "ACCOUNT_TRANSACTION_ID",
    "SIGNERS"
]

transactions_clean = transactions.drop(
    columns=columns_to_drop
).copy()

print("Before:", transactions.shape)
print("After:", transactions_clean.shape)
print("Missing values:", transactions_clean.isna().sum().sum())

Before: (40131, 26)
After: (40131, 19)
Missing values: 0


In [6]:
print("=== DUPLICATES ===")

print("Duplicate transaction hashes:",
      transactions_clean["TRANSACTION_HASH"].duplicated().sum())

print("Duplicate label hashes:",
      labels["TRANSACTION_HASH"].duplicated().sum())

print("\n=== TARGET ===")

print("Missing target values:",
      labels["is_suspicious"].isna().sum())

print("\nTarget distribution:")
print(labels["is_suspicious"].value_counts())

print("\nTarget percentages:")
print(labels["is_suspicious"].value_counts(normalize=True) * 100)

=== DUPLICATES ===
Duplicate transaction hashes: 0
Duplicate label hashes: 0

=== TARGET ===
Missing target values: 0

Target distribution:
is_suspicious
0    38881
1     1250
Name: count, dtype: int64

Target percentages:
is_suspicious
0    96.885201
1     3.114799
Name: proportion, dtype: float64
